# Frequency and Keyness

## Housekeeping (no interaction required)

In [1]:
%pip install simplemma
%pip install nltk

/home/bode-wsl/projects/summerschool26/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/bode-wsl/projects/summerschool26/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
import time
from pathlib import Path

import pandas as pd
import nltk
import simplemma
from tqdm.notebook import tqdm

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /home/bode-
[nltk_data]     wsl/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBSummerSchool26/data') if IN_COLAB else Path('../data')
ZB_MODULE = '03' # Identifier of the ZB Summer School module, used for naming output files

## Setup (Interaction required)

In [4]:
### ⬇️⬇️⬇️ 💽 Adjust here if you want to continue with your own query
CORPUS_NAME = "armensteuer_and_similars"
### ⬆️⬆️⬆️

💽 You only need to run the cell below if you want to work with your own query.

*Once prompted, give all demanded permissions*

In [5]:
# 💽
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

## Load the data


### <img src="https://cdn.simpleicons.org/googledrive" alt="💾" width=16> Load your own data from Google Drive

### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [6]:
RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.02.parquet" # Use data from filtering module
raw_df = pd.read_parquet(RAWDATA_PATH)

FileNotFoundError: [Errno 2] No such file or directory: '../data/armensteuer_and_similars.02.parquet'

### <img src="https://cdn.simpleicons.org/github" alt="🏫" width=16> Load from Github

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from example

In [6]:
RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.01.parquet" # CHANGE TO 02
raw_df = pd.read_parquet(RAWDATA_ORIGIN_URL)

⚙️ Only for development, delete for summer school!

In [7]:
raw_df = raw_df.reset_index()[["id", "meta.date", "meta.mediaTitle", "text.itemTypeLabel", "text.content", "text.contentLength"]]
raw_df = raw_df.set_index("id")

### Parse data

In [8]:
raw_df["year"] = pd.to_datetime(raw_df["meta.date"]).dt.year

## Preprocess Corpus

In [9]:
def sentencize(s: str) -> list[str]:
    sentences = nltk.tokenize.sent_tokenize(s, language="german")
    return sentences

def tokenize(s: str) -> list[str]:
    tokens = nltk.tokenize.word_tokenize(s, language="german")
    return tokens

def lemmatize(s: list[str]) -> list[str]:
    lemmatized = [simplemma.lemmatize(word, lang="de") for word in s]
    return lemmatized

tqdm.pandas(desc="Applying sentencization")
raw_df["_sentences"] = raw_df["text.content"].progress_apply(sentencize)

tqdm.pandas(desc="Applying tokenization")
raw_df["tokens"] = raw_df["_sentences"].progress_apply(lambda sentences: [tokenize(sentence) for sentence in sentences])

tqdm.pandas(desc="Applying lemmatization")
raw_df["lemmas"] = raw_df["tokens"].progress_apply(lambda tokens: [lemmatize(token_list) for token_list in tokens])


Applying sentencization:   0%|          | 0/4044 [00:00<?, ?it/s]

Applying tokenization:   0%|          | 0/4044 [00:00<?, ?it/s]

Applying lemmatization:   0%|          | 0/4044 [00:00<?, ?it/s]

## Frequency Analyses

In [10]:
from typing import Generator 

def token_stream(df, column: str) -> Generator[str, None, None]:
    for doc in df[column]:
        for sentence in doc:
            for token in sentence:
                yield token

print("The `token_stream` generator yields tokens one by one:")
for i, token in enumerate(token_stream(raw_df, "tokens")):
    if i >= 10:  # Print only the first 10 tokens for demonstration
        break
    print(token, end='   ')
print("...")

The `token_stream` generator yields tokens one by one:
KOn   «   öß5   2s   .   Ml   »   7   ?   l   ...


In [ ]:
from collections import Counter
# Compare Counter with pure pandas approach

def count_tokens(df, column: str) -> Counter:
    counter = Counter()

    for doc in df[column]:
        for sentence in doc:
            for token in sentence:
                counter[token] += 1
    
    return counter



Counter approach took 3.5949 seconds
Pandas approach took 19.4970 seconds


In [ ]:
from collections import Counter
import math

def count_tokens(df, column: str) -> Counter:
    counter = Counter()
    for doc in tqdm(df[column], desc="Counting Tokens", leave=False):
        for sentence in doc:
            for token in sentence:
                counter[token] += 1
    return counter

class KeynessComparer:
    def __init__(self, df1: pd.DataFrame, df2: pd.DataFrame, column: str = "lemmas"):
        self.df1 = df1
        self.df2 = df2
        self.column = column

        self.counter1 = count_tokens(self.df1, self.column)
        self.counter2 = count_tokens(self.df2, self.column)

        self.total_tokens1 = sum(self.counter1.values())
        self.total_tokens2 = sum(self.counter2.values())

    def build_contingency_df(self) -> pd.DataFrame:
        all_tokens = set(self.counter1.keys()) | set(self.counter2.keys())
        data = []
        for token in all_tokens:
            observed_tok_1 = self.counter1[token]
            observed_tok_2 = self.counter2[token]
            observed_notok_1 = self.total_tokens1 - observed_tok_1
            observed_notok_2 = self.total_tokens2 - observed_tok_2

            # Expected frequencies under the null hypothesis of no association between token and corpus
            expected_tok_1 = (observed_tok_1 + observed_tok_2) * self.total_tokens1 / (self.total_tokens1 + self.total_tokens2)
            expected_tok_2 = (observed_tok_1 + observed_tok_2) * self.total_tokens2 / (self.total_tokens1 + self.total_tokens2)
            expected_notok_1 = (observed_notok_1 + observed_notok_2) * self.total_tokens1 / (self.total_tokens1 + self.total_tokens2)
            expected_notok_2 = (observed_notok_1 + observed_notok_2) * self.total_tokens2 / (self.total_tokens1 + self.total_tokens2)

            data.append({
                "token": token,
                "observed_tok_1": observed_tok_1,
                "observed_tok_2": observed_tok_2,
                "observed_notok_1": observed_notok_1,
                "observed_notok_2": observed_notok_2,
                "expected_tok_1": expected_tok_1,
                "expected_tok_2": expected_tok_2,
                "expected_notok_1": expected_notok_1,
                "expected_notok_2": expected_notok_2,
            })
        contingency_df = pd.DataFrame(data)
        return contingency_df

    def log_likelihood_ratio(self, contingency_row: pd.Series) -> float:
        G = 0.0
        for token_presence in ["tok", "notok"]:
            for corpus in ["1", "2"]:
                observed = contingency_row[f"observed_{token_presence}_{corpus}"]
                expected = contingency_row[f"expected_{token_presence}_{corpus}"]

                if observed > 0 and expected > 0:
                    G += observed * math.log(observed / expected)
        return 2 * G
                

    def log_ratio(self, contingency_row: pd.Series) -> float:
        observed_tok_1 = contingency_row["observed_tok_1"]
        observed_tok_2 = contingency_row["observed_tok_2"]

        corpus1 = math.log((observed_tok_1 + 1/2) / (self.total_tokens1 + 1/2))  # Add-one smoothing
        corpus2 = math.log((observed_tok_2 + 1/2) / (self.total_tokens2 + 1/2))  # Add-one smoothing

        return corpus1 - corpus2